In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

d:\Anaconda\envs\tcc_faturas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

tokenizer = AutoTokenizer.from_pretrained(
    "google/byt5-small"
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/byt5-small",
    device_map="auto"
)

input_ids = tokenizer("summarize: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy.", return_tensors="pt").to(model.device)

output = model.generate(**input_ids)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 172/172 [00:00<00:00, 329.29it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
d:\Anaconda\envs\tcc_faturas\Lib\site-packages\transformers\generation\utils.py:1616: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


: Photosynthesis is


In [3]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

: Photosynthesis is


In [19]:
# pip install torchao
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TorchAoConfig
from torchao.quantization import Int4WeightOnlyConfig

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/byt5-xl",
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("google/byt5-xl")
input_ids = tokenizer("translate English to French: The weather is nice today.", return_tensors="pt").to(model.device)

output = model.generate(**input_ids)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 500/500 [00:18<00:00, 27.47it/s] 
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Some parameters are on the meta device because they were offloaded to the cpu.


 weather is nice to


In [20]:
input_ids = tokenizer("Th1s is @ noi$y t3xt th@t ne3ds to be n0rmalized!!!", return_tensors="pt").to(model.device)

output = model.generate(**input_ids)
print(tokenizer.decode(output[0], skip_special_tokens=True))

 the t3xt th@s is @


In [1]:

import torch, gc
from transformers import AutoModelForSequenceClassification, AutoTokenizer

gc.collect()
torch.cuda.empty_cache()

# Categorias de exemplo
labels = ["alimentacao", "transporte", "streaming", "compras", "outros"]

model = AutoModelForSequenceClassification.from_pretrained(
    "google/byt5-large",
    num_labels=len(labels),
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("google/byt5-large")

# Exemplo de descrição de fatura
texto = "UBER* TRIP 12DEC"

inputs = tokenizer(texto, return_tensors="pt", max_length=128, truncation=True).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()

print(f"Categoria: {labels[pred]}")

d:\Anaconda\envs\tcc_faturas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0804 23:24:13.786000 34908 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0804 23:24:14.348000 34908 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 499/499 [03:37<00:00,  2.30it/s]
[transformers] T5ForSequenceCl

Categoria: alimentacao


In [2]:
novo_texto = "UBER* TRIP 12DEC"

inputs = tokenizer(novo_texto, return_tensors="pt", max_length=128, truncation=True).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()

print(f"Categoria: {labels[pred]}") 

Categoria: alimentacao


### 1. Primeiro experimento traduzir transacoes de faturas 

In [1]:
%env KAGGLE_KEY=KGAT_345ff9f1af4cf2fa28ea0a3cab3fd477
%env KAGGLE_USERNAME=luizmonteiro
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealtman2019/credit-card-transactions")

print("Path to dataset files:", path)

env: KAGGLE_KEY=KGAT_345ff9f1af4cf2fa28ea0a3cab3fd477
env: KAGGLE_USERNAME=luizmonteiro


d:\Anaconda\envs\tcc_faturas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\DEVELOPER\.cache\kagglehub\datasets\ealtman2019\credit-card-transactions\versions\8


In [5]:
import pandas as pd
data = pd.read_csv(f'{path}/User0_credit_card_transactions.csv')
data.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [13]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 4380.13it/s]


In [ ]:

categorias_gastos = [
    # Alimentação
    "Restaurantes",
    "Fast Food",
    "Padarias e Confeitarias",
    "Supermercados",
    "Delivery de Comida",
    "Bares e Lanchonetes",

    # Transporte
    "Combustível",
    "Estacionamento",
    "Pedágio",
    "Transporte por Aplicativo (Uber/99)",
    "Táxi",
    "Ônibus / Metrô",
    "Manutenção Veicular",
    "Seguro de Veículo",
    "IPVA",

    # Moradia
    "Aluguel",
    "Condomínio",
    "Energia Elétrica",
    "Água e Esgoto",
    "Gás",
    "Internet",
    "Telefone Fixo",
    "TV por Assinatura",
    "Materiais de Limpeza",
    "Manutenção / Reformas",

    # Saúde
    "Plano de Saúde",
    "Farmácias / Medicamentos / Drogaria",
    "Consultas Médicas",
    "Exames Laboratoriais",
    "Dentista",
    "Psicólogo / Terapia",
    "Academia / Fitness",
    "Ótica",

    # Educação
    "Mensalidade Escolar",
    "Mensalidade Universitária",
    "Cursos Online",
    "Livros e Material Didático",
    "Papelaria",

    # Vestuário
    "Roupas",
    "Calçados",
    "Acessórios",

    # Lazer e Entretenimento
    "Cinema / Teatro",
    "Shows e Eventos",
    "Viagens e Hospedagem",
    "Passagens Aéreas",
    "Streaming (Netflix, Spotify...)",
    "Jogos e Aplicativos",
    "Hobbies",

    # Tecnologia
    "Eletrônicos",
    "Celular / Plano Móvel",
    "Softwares e Assinaturas",
    "Equipamentos de Informática",

    # Beleza e Cuidados Pessoais
    "Salão de Beleza / Barbearia",
    "Cosméticos e Perfumaria",
    "Higiene Pessoal",

    # Pets
    "Alimentação Animal",
    "Veterinário",
    "Pet Shop",

    # Finanças
    "Seguros",
    "Investimentos",
    "Empréstimos / Financiamentos",
    "Taxas e Tarifas Bancárias",
    "Impostos (IPTU, IR...)",

    # Compras Online / Marketplace
    "E-commerce (Amazon, Mercado Livre...)",
    "Assinaturas de Clube",

    # Doações e Presentes
    "Doações",
    "Presentes",

    # Outros
    "Serviços Domésticos",
    "Cartório e Documentos",
    "Multas",
    "Despesas Diversas",
]

def classificar_fatura(texto):
    resultado = classifier(
        f"Compra realizada no estabelecimento: {texto}",
        candidate_labels=categorias_gastos,
        hypothesis_template="Esta despesa pertence à categoria {}."
    )
    return {
        "categoria": resultado["labels"][0],
        "confiança": f"{resultado['scores'][0]:.1%}"
    }

def classificar_lote(textos):
    resultados = classifier(
            f"Compra realizada no estabelecimento: {textos}",
            candidate_labels=categorias_gastos,
            hypothesis_template="Esta despesa pertence à categoria {}.",
            batch_size=32
        )
    return [
        {
            "categoria": resultado["labels"][0],
            "confiança": f"{resultado['scores'][0]:.1%}"
        }
        for resultado in resultados
    ]

In [6]:
textos = [
    "UBER* VIAGEM 12DEZ",
    "NETFLIX.COM",
    "IFOOD*RESTAURANTE",
    "AMZN*MARKETPLACE",
    "SP TIM Black C Light 9 0",
]

resultados = classificar_lote(textos)
for texto, resultado in zip(textos, resultados):
    print(f"{texto:<30} → {resultado['categoria']} ({resultado['confiança']})")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


UBER* VIAGEM 12DEZ             → transporte (61.4%)
NETFLIX.COM                    → streaming (57.4%)
IFOOD*RESTAURANTE              → alimentacao (91.9%)
AMZN*MARKETPLACE               → compras (61.9%)
SP TIM Black C Light 9 0       → Serviços de telecomunicações (47.7%)


### Gerar os dados com ruido

In [1]:
import random
import string

def simular_nome_maquininha_cartao(nome, comprimento_max=20):
    """
    Simula como o nome de um fornecedor aparece em uma descrição de máquina de cartão.
    Adiciona ruído realista como truncamento, códigos e caracteres estranhos.
    
    Parameters:
    -----------
    nome : str
        Nome original do fornecedor
    comprimento_max : int
        Comprimento máximo permitido pela máquina
    
    Returns:
    --------
    str
        Nome com ruído simulado
    """
    
    if pd.isna(nome):
        return nome
    
    nome_str = str(nome).upper().strip()
    
    truncado = nome_str[:comprimento_max]
    
    tipos_ruido = random.choices(
        [0, 1, 2, 3, 4],
        weights=[30, 25, 20, 15, 10],
        k=1
    )[0]
    
    if tipos_ruido == 0:
        sufixo = f" {random.randint(100, 9999)}"
        resultado = (truncado[:comprimento_max-len(sufixo)] + sufixo).rstrip()
    
    elif tipos_ruido == 1:
        prefixo_loja = f"*{random.randint(10, 9999)} "
        resultado = (prefixo_loja + truncado[:comprimento_max-len(prefixo_loja)]).rstrip()
    
    elif tipos_ruido == 2:
        partes = truncado.split()
        if len(partes) > 1:
            resultado = partes[0] + " " + partes[1][:3].upper()
        else:
            resultado = truncado
    
    elif tipos_ruido == 3:
        caracteres_aleatorios = ''.join(random.choices(string.ascii_uppercase + string.digits, k=3))
        resultado = truncado[:comprimento_max-4] + caracteres_aleatorios
    
    else:
        resultado = truncado
    
    return resultado.strip()

def gerar_dataset_com_ruido(df, coluna_nome='NOME FAVORECIDO', probabilidade=1.0):
    """
    Adiciona coluna com nomes simulados de máquina de cartão.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame com os dados
    coluna_nome : str
        Nome da coluna com os fornecedores
    probabilidade : float
        Probabilidade de adicionar ruído (0-1)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame com nova coluna 'DESCRICAO_MAQUININHA'
    """
    
    df_novo = df.copy()
    
    df_novo['DESCRICAO_MAQUININHA'] = df_novo[coluna_nome].apply(
        lambda x: simular_nome_maquininha_cartao(x) if random.random() < probabilidade else x
    )
    
    print(f"✅ Coluna 'DESCRICAO_MAQUININHA' criada com ruído de máquina de cartão!")
    print(f"\nExemplos de transformação:")
    
    comparacao = df_novo[[coluna_nome, 'DESCRICAO_MAQUININHA']].drop_duplicates().head(10)
    for idx, row in comparacao.iterrows():
        print(f"  {row[coluna_nome]:30s} → {row['DESCRICAO_MAQUININHA']:25s}")
    
    return df_novo

print("Função de simulação de máquina de cartão criada!")
print("\nUso:")
print("df_com_ruido = gerar_dataset_com_ruido(df_combinado)")
print("\nOu com probabilidade customizada:")
print("df_com_ruido = gerar_dataset_com_ruido(df_combinado, probabilidade=0.8)")

Função de simulação de máquina de cartão criada!

Uso:
df_com_ruido = gerar_dataset_com_ruido(df_combinado)

Ou com probabilidade customizada:
df_com_ruido = gerar_dataset_com_ruido(df_combinado, probabilidade=0.8)


In [2]:
import pandas as pd
faturas_df = pd.read_csv('../data/datasets/faturas_gov.csv')

C:\Users\DEVELOPER\AppData\Local\Temp\ipykernel_15424\3176924632.py:2: DtypeWarning: Columns (15,16,17,18,19,20,21,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  faturas_df = pd.read_csv('../data/datasets/faturas_gov.csv')


In [5]:
faturas_sinteticas = gerar_dataset_com_ruido(faturas_df)

✅ Coluna 'DESCRICAO_MAQUININHA' criada com ruído de máquina de cartão!

Exemplos de transformação:
  IMPORTADORA OPLIMA LTDA        → *3761 IMPORTADORA OP     
  JOSE PEREIRA DE SOUZA MOLDURAS → JOSE PEREIRA DE PZ4      
  EXTINTORES ARAGUAIA LTDA       → EXTINTORES ARAGU 270     
  SACARIA ESTRELA LTDA           → *4684 SACARIA ESTREL     
  BIG CHAVES COMERCIO E SERVICOS DE CHAVES, CARIMBOS E SISTEMA DE SEGURANCA LTDA → BIG CHAVES COME 6322     
  JANIO C. C. DA SILVA           → JANIO C. C. DA  2504     
  IG PLOTTER GRAFICA RAPIDA LTDA → IG PLOTTER GRAF 1346     
  FUJIOKA ELETRO IMAGEM S.A      → FUJIOKA ELETRO II30      
  OPCAO FERRAZ FERRAGISTA LTDA   → OPCAO FERRAZ FER 744     
  OPCAO FERRAZ FERRAGISTA LTDA   → OPCAO FERRAZ FERGOG      


In [6]:
faturas_sinteticas.describe()

,CÓDIGO ÓRGÃO SUPERIOR,CÓDIGO ÓRGÃO,CÓDIGO UNIDADE GESTORA,ANO EXTRATO,MÊS EXTRATO,CNPJ OU CPF FAVORECIDO
count,476699.000000,476699.000000,476699.000000,476699.000000,476699.000000,4.766990e+05
mean,35857.449728,31667.093264,208290.787067,2024.239791,7.016012,1.542678e+13
std,11351.031770,10526.563917,137609.435439,0.969557,3.393850,2.259966e+13
min,20000.000000,20101.000000,110001.000000,2023.000000,1.000000,-1.100000e+01
25%,26000.000000,25205.000000,135048.000000,2023.000000,4.000000,-2.000000e+00
50%,30000.000000,26438.000000,160410.000000,2024.000000,7.000000,4.587328e+12
75%,47000.000000,32396.000000,200372.000000,2025.000000,10.000000,2.369226e+13
max,81000.000000,81000.000000,888310.000000,2026.000000,12.000000,9.874986e+13


In [18]:
faturas_sinteticas['NOME FAVORECIDO']
for idx,row in faturas_sinteticas.sample(n=20,replace=True).iterrows():
    resultado = classificar_fatura(row['NOME FAVORECIDO'])
    print(f"{row['DESCRICAO_MAQUININHA']:<30} → {resultado['categoria']} ({resultado['confiança']}) Estabelecimento:{row['NOME FAVORECIDO']}")

*6172 SEM INFORMACAO           → Multas (14.0%) Estabelecimento:SEM INFORMACAO
*306 LOTUFO & BARBOS           → Salão de Beleza / Barbearia (19.2%) Estabelecimento:LOTUFO & BARBOSA LTDA
NAO SE APLICA 724              → Aluguel (3.7%) Estabelecimento:NAO SE APLICA
SEM INFORMACAO 5584            → Multas (14.0%) Estabelecimento:SEM INFORMACAO
SIGILOSO                       → Seguros (9.5%) Estabelecimento:Sigiloso
NAO SE APLICA                  → Aluguel (3.7%) Estabelecimento:NAO SE APLICA
NAO SE APLICA                  → Aluguel (3.7%) Estabelecimento:NAO SE APLICA
*4580 PRIMEIRA LINHA           → Passagens Aéreas (12.8%) Estabelecimento:PRIMEIRA LINHA COMERCIAL DE ROLAMENTOS LTDA
SIGILOSO                       → Seguros (9.5%) Estabelecimento:Sigiloso
SIGILOSO 6635                  → Seguros (9.5%) Estabelecimento:Sigiloso
*7820 SANTO ANTONIO            → Manutenção Veicular (31.3%) Estabelecimento:SANTO ANTONIO AUTOPECAS LTDA
POSTO DE COMBUS 7938           → Combustível (39.4%) Estab

In [ ]:
from openrouter import OpenRouter
import os

with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
    response = client.chat.send(
        model="openai/GPT-4o-mini",
        messages=[
            {"role": "user", "content": "Explain quantum computing in one sentence."}
        ],
    )

    print(response.choices[0].message.content) 


Quantum computing is a type of computation that leverages the principles of quantum mechanics, using qubits to perform operations on data in ways that classical computers cannot, potentially solving certain problems much faster.


: 

In [2]:
import pandas as pd
df = pd.read_csv('../data/datasets/faturas_nubank.csv')
df.head()

,date,title,amount
0,2023-12-30,Tim*Tim,"40,00"
1,2023-12-29,Max,"30,00"
2,2023-12-20,Pag*Principia,"200,45"
3,2023-12-15,Pagamento recebido,"- 421,04"
4,2023-12-13,Ogura Pasteis,"20,00"


In [11]:
sample_df = df.iloc[1:100]
textos_amostra = sample_df['title'].tolist()

resultados = classificar_lote(textos_amostra)
for (idx, row), resultado in zip(sample_df.iterrows(), resultados):
    print(f"{row['title']:<30} → {resultado['categoria']} ({resultado['confiança']})")

Max                            → Multas (10.6%)
Pag*Principia                  → Papelaria (7.5%)
Pagamento recebido             → Pedágio (22.1%)
Ogura Pasteis                  → Padarias e Confeitarias (12.0%)
Drogalis Italo - Parcela 2/2   → Cosméticos e Perfumaria (9.1%)
Oggi Sorvetes Aruja            → Passagens Aéreas (11.6%)
Suzan Bela - Parcela 5/5       → Presentes (5.8%)
Padaria e Conveniencia         → Padarias e Confeitarias (22.7%)
Vitor Moreira Cardoso          → Presentes (3.5%)
Max                            → Multas (10.6%)
Creamy - Parcela 1/3           → Materiais de Limpeza (6.3%)
Bazar Xereta                   → Materiais de Limpeza (5.3%)
Tim*Tim                        → Gás (4.0%)
I9pay S*Supermercado T         → Supermercados (57.2%)
I9pay S*Supermercado T         → Supermercados (57.2%)
I9pay S*Supermercado T         → Supermercados (57.2%)
Drogalis Italo                 → Cosméticos e Perfumaria (9.6%)
Fisia Nike Ecommer - Parcela 1/2 → Condomínio (6.9%)
I9pay

In [12]:
for idx, row in sample_df.iterrows():
    resultado = classificar_fatura(row['title'])
    print(f"{row['title']:<30} → {resultado['categoria']} ({resultado['confiança']})")

Max                            → Multas (10.6%)
Pag*Principia                  → Papelaria (7.5%)
Pagamento recebido             → Pedágio (22.1%)
Ogura Pasteis                  → Padarias e Confeitarias (12.0%)
Drogalis Italo - Parcela 2/2   → Cosméticos e Perfumaria (9.1%)


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Oggi Sorvetes Aruja            → Passagens Aéreas (11.6%)
Suzan Bela - Parcela 5/5       → Presentes (5.8%)
Padaria e Conveniencia         → Padarias e Confeitarias (22.7%)
Vitor Moreira Cardoso          → Presentes (3.5%)
Max                            → Multas (10.6%)
Creamy - Parcela 1/3           → Materiais de Limpeza (6.3%)
Bazar Xereta                   → Materiais de Limpeza (5.3%)
Tim*Tim                        → Gás (4.0%)
I9pay S*Supermercado T         → Supermercados (57.2%)
I9pay S*Supermercado T         → Supermercados (57.2%)
I9pay S*Supermercado T         → Supermercados (57.2%)
Drogalis Italo                 → Cosméticos e Perfumaria (9.6%)
Fisia Nike Ecommer - Parcela 1/2 → Condomínio (6.9%)
I9pay S*Supermercado T         → Supermercados (57.2%)
Estorno de "Produtos Globo"    → Assinaturas de Clube (6.3%)
Pagamento recebido             → Pedágio (22.1%)
Produtos Globo                 → Eletrônicos (8.0%)
Ebn *Spotify                   → Shows e Eventos (9.9%)
Tim*Tim 

KeyboardInterrupt: 